# Stage 6.1 – Automated Evaluation of the Frozen A/B/C Reports

This notebook compares all 45 reports using the same automated measures. It does not generate or rewrite any report.

Important points:

- A, B and C are compared against the same frozen PMC passages for each question.
- Similarity values are not scores out of five and are not proof of medical correctness.
- Citation and claim-verification results are kept separate because only Condition C was designed to cite PMC evidence.


## 1. Install the fixed libraries

I pin the package versions so the evaluation can be reproduced later. A GPU is optional for this notebook.


In [ ]:
!pip -q install "sentence-transformers==3.4.1" "scipy==1.15.2"


## 2. Upload and verify the frozen evaluation input

Upload `Adarsh_Konderu_Stage_6_Frozen_Evaluation_Input.zip` when Colab asks for it. The notebook checks the archive hash before using the data.


In [ ]:
from google.colab import files
from pathlib import Path
import csv
import hashlib
import json
import math
import re
import shutil
import statistics
import time
import zipfile

EXPECTED_INPUT_SHA256 = "d47dadcef96fa2f492fb05d6876ca0f3bd851c4b3583b090a6ba39557517513e"
uploaded = files.upload()
input_name = next(iter(uploaded))
input_bytes = uploaded[input_name]
actual_hash = hashlib.sha256(input_bytes).hexdigest()

print("Expected SHA-256:", EXPECTED_INPUT_SHA256)
print("Uploaded SHA-256:", actual_hash)
assert actual_hash == EXPECTED_INPUT_SHA256, "The uploaded input is not the frozen Stage 6 file."

work_dir = Path("/content/stage_6_1_work")
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)

with zipfile.ZipFile(input_name) as archive:
    archive.extractall(work_dir)

input_dir = work_dir / "stage_6_frozen_evaluation_input"
cases = [json.loads(line) for line in (input_dir / "evaluation_cases.jsonl").read_text().splitlines() if line.strip()]
assert len(cases) == 15
assert sum(len(case["reports"]) for case in cases) == 45
print("Verified 15 questions and 45 frozen reports.")


## 3. Load the fixed embedding model

The same sentence-transformer used in the retrieval work is used for every report. Normalised embeddings allow cosine similarity to be calculated with a dot product.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"

model = SentenceTransformer(EMBEDDING_MODEL, revision=EMBEDDING_REVISION)
print("Loaded:", EMBEDDING_MODEL)
print("Revision:", EMBEDDING_REVISION)


## 4. Define transparent metric functions

The report is divided into sentences. Evidence support asks how closely each report sentence matches its best reference passage. Evidence coverage asks how closely each reference passage is represented by its best report sentence.


In [ ]:
def plain_text(markdown_text):
    """Remove Markdown markers for language-level measurements."""
    text = re.sub(r"^#{1,6}\s+", "", markdown_text, flags=re.MULTILINE)
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)
    text = re.sub(r"[*_`>-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def split_sentences(text):
    """Use a simple reproducible sentence boundary rule."""
    clean = plain_text(text)
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", clean)
    return [part.strip() for part in parts if len(part.split()) >= 4]


def count_sections(text):
    """Count level-two Markdown headings used as report sections."""
    return len(re.findall(r"^##\s+.+$", text, flags=re.MULTILINE))


def lexical_diversity(text):
    """Calculate lower-case type-token ratio."""
    tokens = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", plain_text(text).lower())
    return len(set(tokens)) / len(tokens) if tokens else float("nan")


def estimated_syllables(word):
    """Estimate syllables without relying on an external pronunciation dictionary."""
    word = re.sub(r"[^a-z]", "", word.lower())
    if not word:
        return 0
    groups = len(re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and not word.endswith(("le", "ye")) and groups > 1:
        groups -= 1
    return max(1, groups)


def flesch_reading_ease(text):
    """Calculate Flesch Reading Ease using a documented syllable estimate."""
    clean = plain_text(text)
    words = re.findall(r"[A-Za-z]+", clean)
    sentences = split_sentences(clean)
    if not words or not sentences:
        return float("nan")
    syllables = sum(estimated_syllables(word) for word in words)
    return 206.835 - 1.015 * (len(words) / len(sentences)) - 84.6 * (syllables / len(words))


def encode(items):
    return model.encode(items, normalize_embeddings=True, show_progress_bar=False)


def calculate_semantic_metrics(question, report, references):
    sentences = split_sentences(report)
    reference_texts = [item["text"] for item in references]
    question_vector = encode([question])[0]
    report_vector = encode([plain_text(report)])[0]
    sentence_vectors = encode(sentences)
    reference_vectors = encode(reference_texts)

    similarity_matrix = sentence_vectors @ reference_vectors.T
    return {
        "question_relevance_cosine": float(question_vector @ report_vector),
        "evidence_support_mean": float(similarity_matrix.max(axis=1).mean()),
        "evidence_coverage_mean": float(similarity_matrix.max(axis=0).mean()),
    }


## 5. Calculate one row per frozen report

This cell may take several minutes because it embeds every report and reference passage.


In [ ]:
rows = []
started = time.time()

for case_number, case in enumerate(cases, start=1):
    for report in case["reports"]:
        # Common content metrics use the citation-blinded view for every
        # condition. This removes Condition C's source-list formatting from
        # the comparison while preserving its original citation audit below.
        report_text = report["judge_view_text"]
        clean = plain_text(report_text)
        words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", clean)
        metric_row = {
            "question_id": case["question_id"],
            "theme": case["theme"],
            "condition": report["condition"],
            "word_count": len(words),
            "sentence_count": len(split_sentences(report_text)),
            "section_count": count_sections(report_text),
            "type_token_ratio": lexical_diversity(report_text),
            "flesch_reading_ease": float(flesch_reading_ease(clean)),
            "generation_seconds": report["generation_record"]["duration_seconds"],
            "structure_passed": report["generation_record"]["structure_passed"],
            "citation_count": report["generation_record"]["citation_count"],
            "claim_support_rate": report["generation_record"]["claim_support_rate"],
        }
        metric_row.update(calculate_semantic_metrics(case["question"], report_text, case["references"]))
        rows.append(metric_row)
    print(f"Completed question {case_number}/15: {case['question_id']}")

metrics = pd.DataFrame(rows).sort_values(["question_id", "condition"]).reset_index(drop=True)
assert len(metrics) == 45
print(f"Finished in {(time.time() - started) / 60:.1f} minutes")
display(metrics.head(9))


## 6. Summarise by condition

No winner is selected in this cell. It reports each measure separately so the trade-offs remain visible.


In [ ]:
analysis_metrics = [
    "question_relevance_cosine",
    "evidence_support_mean",
    "evidence_coverage_mean",
    "flesch_reading_ease",
    "type_token_ratio",
    "word_count",
    "section_count",
    "generation_seconds",
]

summary = metrics.groupby("condition")[analysis_metrics].agg(["mean", "std", "median"]).round(4)
display(summary)


## 7. Paired statistical tests

The same 15 questions were answered under all three conditions, so this is a repeated-measures comparison. The Friedman test is followed by paired Wilcoxon tests. Holm correction controls the three pairwise comparisons for each metric.


In [ ]:
def holm_adjust(p_values):
    """Return Holm-adjusted p-values in their original order."""
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running_max = 0.0
    for rank, index in enumerate(order):
        candidate = min(1.0, (len(p_values) - rank) * p_values[index])
        running_max = max(running_max, candidate)
        adjusted[index] = running_max
    return adjusted


def rank_biserial_from_differences(differences):
    """Matched-pairs rank-biserial effect size; positive favours the first condition."""
    differences = np.asarray(differences, dtype=float)
    differences = differences[differences != 0]
    if len(differences) == 0:
        return 0.0
    ranks = pd.Series(np.abs(differences)).rank(method="average").to_numpy()
    positive = ranks[differences > 0].sum()
    negative = ranks[differences < 0].sum()
    return float((positive - negative) / (positive + negative))


condition_order = ["A", "B", "C"]
pair_order = [("A", "B"), ("A", "C"), ("B", "C")]
test_rows = []

for metric in analysis_metrics:
    wide = metrics.pivot(index="question_id", columns="condition", values=metric)[condition_order].dropna()
    friedman_stat, friedman_p = friedmanchisquare(wide["A"], wide["B"], wide["C"])
    pair_results = []
    for first, second in pair_order:
        differences = wide[first] - wide[second]
        # SciPy cannot run the default Wilcoxon calculation when every
        # paired difference is zero, so that exact tie is recorded as p=1.
        if np.allclose(differences, 0):
            statistic, raw_p = 0.0, 1.0
        else:
            result = wilcoxon(wide[first], wide[second], zero_method="wilcox", alternative="two-sided")
            statistic, raw_p = float(result.statistic), float(result.pvalue)
        pair_results.append({
            "comparison": f"{first}-{second}",
            "wilcoxon_statistic": statistic,
            "p_raw": raw_p,
            "rank_biserial_first_minus_second": rank_biserial_from_differences(differences),
        })
    adjusted = holm_adjust([item["p_raw"] for item in pair_results])
    for item, adjusted_p in zip(pair_results, adjusted):
        test_rows.append({
            "metric": metric,
            "n_questions": len(wide),
            "friedman_statistic": float(friedman_stat),
            "friedman_p": float(friedman_p),
            **item,
            "p_holm": float(adjusted_p),
        })

statistical_tests = pd.DataFrame(test_rows)
display(statistical_tests.round(6))


## 8. Save the evidence package

The output contains report-level metrics, descriptive summaries, statistical tests, the executed protocol and a trace with package versions. Download the ZIP and keep it with the dissertation evidence.


In [ ]:
import importlib.metadata
import platform

output_dir = Path("/content/stage_6_1_automated_evaluation_evidence")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True)

metrics.to_csv(output_dir / "automated_metrics_by_report.csv", index=False)
summary.to_csv(output_dir / "automated_metrics_descriptive_summary.csv")
statistical_tests.to_csv(output_dir / "automated_metrics_statistical_tests.csv", index=False)
shutil.copy(input_dir / "STAGE_6_EVALUATION_PROTOCOL.md", output_dir / "STAGE_6_EVALUATION_PROTOCOL.md")

trace = {
    "stage": "Stage 6.1 automated evaluation",
    "input_archive_sha256": actual_hash,
    "question_count": 15,
    "report_count": 45,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_revision": EMBEDDING_REVISION,
    "python_version": platform.python_version(),
    "package_versions": {
        name: importlib.metadata.version(name)
        for name in ["sentence-transformers", "scipy", "pandas", "numpy"]
    },
    "interpretation_warning": "Similarity is not a five-point score and does not prove factual correctness.",
}
(output_dir / "stage_6_1_trace.json").write_text(json.dumps(trace, indent=2))

checksums = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(output_dir.iterdir()) if path.is_file()
}
(output_dir / "checksums_sha256.json").write_text(json.dumps(checksums, indent=2))

archive_base = "/content/Adarsh_Konderu_Stage_6_1_Automated_Evaluation_Evidence"
archive_path = shutil.make_archive(archive_base, "zip", output_dir.parent, output_dir.name)
print("Saved evidence package:", archive_path)
files.download(archive_path)
